# Multimodal AI Fundamentals + Agent Workflow Consolidation
### Practice Notebook (Deliverable Work Session)

**Assumed pre-installed libraries:** none required for Parts 1-2 (pure
Python). If you want to try a real multimodal API call in Part 3, you'll
need the `anthropic` package and an API key -- clearly marked and skippable.


## 1. Multimodal models: the conceptual bridge from Week 1

Recall from Week 1: text is tokenized into sub-word units, then each token
is mapped to a dense embedding vector, and the Transformer's attention
mechanism lets every token attend to every other token.

A vision-language model extends this same idea: an image is cut into
patches, and each patch is converted into a vector "token" that lives in
(roughly) the same embedding space as text tokens. This lets the *same*
attention mechanism attend across text tokens and image-patch tokens
together -- no fundamentally new architecture is required, just a new kind
of token.


In [ ]:
def tokenize_text(text: str) -> list:
    """Simplified stand-in for text tokenization (Week 1, Day 4)."""
    return text.split()

def tokenize_image_patches(image_width: int, image_height: int, patch_size: int = 16) -> list:
    """Simplified stand-in for how a vision-language model turns an image
    into a sequence of patch 'tokens'. Returns a list of (row, col) patch
    coordinates -- in a real model each of these becomes an embedding vector,
    the same way each word becomes an embedding vector.
    """
    patches = []
    for row in range(0, image_height, patch_size):
        for col in range(0, image_width, patch_size):
            patches.append((row // patch_size, col // patch_size))
    return patches

text_tokens = tokenize_text("What is shown in this chart?")
image_tokens = tokenize_image_patches(image_width=224, image_height=224, patch_size=16)

print(f"Text tokens ({len(text_tokens)}): {text_tokens}")
print(f"Image patch tokens: {len(image_tokens)} total (a 224x224 image at 16x16 patches)")
print(f"First few image patch coordinates: {image_tokens[:5]}")
print()
print(f"Combined sequence length the attention mechanism sees: "
      f"{len(text_tokens) + len(image_tokens)} tokens (text + image)")


**Exercise 5.1:** Recompute `tokenize_image_patches` for a larger image
(e.g., 448x448) with the same `patch_size=16`. How does the number of image
tokens scale as image resolution increases? Given that attention cost grows
with the *square* of sequence length (from Week 1, Day 3), what practical
problem does this suggest for feeding very high-resolution images to a
vision-language model?


## 2. Common multimodal use cases (conceptual walkthrough)

Let's simulate the *shape* of a few common multimodal workflows -- not with
real image processing, but with stand-in functions that show what
information flows where.


In [ ]:
def describe_image_stub(image_description_for_demo: str) -> str:
    """Stand-in for: 'a vision-language model looks at a real image and
    describes it.' We pass in a plain-text description here purely so the
    notebook works without real image files or an API key.
    # TODO: replace with a real multimodal LLM call, passing an actual image.
    """
    return f"(stub) Image shows: {image_description_for_demo}"

def extract_structured_data_from_receipt_stub(receipt_text_for_demo: str) -> dict:
    """Stand-in for: 'a vision-language model reads a scanned receipt image
    and extracts structured fields.'
    # TODO: replace with a real multimodal LLM call + a JSON output schema.
    """
    # Toy parsing logic standing in for real extraction.
    import re
    total_match = re.search(r"Total: Rs\.(\d+)", receipt_text_for_demo)
    return {
        "vendor": receipt_text_for_demo.split(",")[0],
        "total": total_match.group(1) if total_match else None,
    }

print(describe_image_stub("a bar chart showing quarterly revenue rising each quarter"))
print()
print(extract_structured_data_from_receipt_stub("Cafe Coffee Day, 2 items, Total: Rs.340"))


**Exercise 5.2:** For each use case below, briefly state (1-2 sentences)
what the "perceive" input would be, and what a plausible tool the agent
might call *afterward*, based on what it perceived:
1. A visually impaired user asks an app to describe what's in front of their
   phone camera.
2. An expense-reporting tool needs to log a photographed receipt.
3. A computer-use agent needs to click the "Submit" button on a webpage
   screenshot.


## 3. (Optional) A real multimodal + tool-use call

If you have an `GROQ or HF API KEYS`, this is what a real multimodal request
looks like -- combining an image input with the tool-calling mechanics from
Day 2. This cell is optional and illustrative; read it even if you can't run
it.


In [ ]:
import base64, os

image_path = "images/"
with open(image_path, "rb") as f:
    image_b64 = base64.standard_b64encode(f.read()).decode("utf-8")
print(image_b64)
#
# client =  # Todo change client definition
#
# log_expense_tool = {
#     "name": "log_expense",
#     "description": "Log a business expense with a vendor name and total amount.",
#     "input_schema": {
#         "type": "object",
#         "properties": {
#             "vendor": {"type": "string"},
#             "total": {"type": "number"},
#         },
#         "required": ["vendor", "total"],
#     },
# }
# Todo: change the response code as per client used:
# response = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=500,
#     tools=[log_expense_tool],
#     messages=[{
#         "role": "user",
#         "content": [
#             {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64}},
#             {"type": "text", "text": "Log this receipt as an expense."},
#         ],
#     }],
# )
# for block in response.content:
#     if block.type == "tool_use":
#         print("Model wants to call:", block.name, "with input:", block.input)

print("This cell is illustrative -- uncomment, add an image file and API key to run it for real.")


## 4. DELIVERABLE: Simple agent workflow implementation

Now assemble a small end-to-end agent using the building blocks from the
whole week:

- Day 1: a perceive-reason-act loop
- Day 2: at least one tool with a proper schema
- Day 3: at least one workflow pattern (routing, orchestrator-worker, etc.)
- Day 4: at least one reasoning/decomposition or self-reflection step

Below is a starter scaffold implementing a **support-ticket triage agent**.
It's intentionally incomplete -- follow the `# TODO` markers to finish it, or
replace the whole scenario with one of your own choosing.


In [ ]:
# --- Tools (Day 2) ---

def check_customer_account_tool(customer_id: str) -> dict:
    """FAKE tool. # TODO: replace with a real lookup / real LLM-driven call."""
    fake_accounts = {"C001": {"plan": "premium", "issues_this_month": 0},
                      "C002": {"plan": "free", "issues_this_month": 3}}
    return fake_accounts.get(customer_id, {"plan": "unknown", "issues_this_month": 0})

check_customer_account_schema = {
    "name": "check_customer_account_tool",
    "description": "Look up a customer's plan tier and how many issues they've filed this month.",
    "input_schema": {
        "type": "object",
        "properties": {"customer_id": {"type": "string"}},
        "required": ["customer_id"],
    },
}

# --- Routing step (Day 3) ---

def classify_ticket(ticket_text: str) -> str:
    """# TODO: replace with a real LLM call."""
    text = ticket_text.lower()
    if any(w in text for w in ["refund", "charge", "billing"]):
        return "billing"
    if any(w in text for w in ["bug", "crash", "error"]):
        return "technical"
    return "general"

# --- Reasoning / decision step (Day 4) ---

def decide_escalation(category: str, account_info: dict) -> bool:
    """# TODO: replace with a real LLM call reasoning over category + account_info.
    Simple rule for this scaffold: escalate technical issues for premium
    customers, or any customer with 3+ issues this month.
    """
    if account_info.get("plan") == "premium" and category == "technical":
        return True
    if account_info.get("issues_this_month", 0) >= 3:
        return True
    return False

# --- The agent loop (Day 1) ---

def triage_agent(ticket_text: str, customer_id: str) -> dict:
    # PERCEIVE
    print(f"[PERCEIVE] New ticket from {customer_id}: '{ticket_text}'")

    # REASON (classify)
    category = classify_ticket(ticket_text)
    print(f"[REASON] Classified as: {category}")

    # ACT (tool call)
    print("[ACT] Calling check_customer_account_tool...")
    account_info = check_customer_account_tool(customer_id)
    print(f"[PERCEIVE] Tool result: {account_info}")

    # REASON (decide escalation) -- TODO: add a self-reflection step here:
    # after deciding, have the agent double-check its own decision against
    # a stated policy (Day 4, self-correction) before finalizing.
    escalate = decide_escalation(category, account_info)
    print(f"[REASON] Escalate to human? {escalate}")

    # ACT (final)
    outcome = {
        "ticket": ticket_text,
        "category": category,
        "account_info": account_info,
        "escalated": escalate,
    }
    print(f"[ACT] Final routing decision: {outcome}")
    return outcome

triage_agent("The app keeps crashing when I try to check out.", "C001")


**TODO for your deliverable:**
1. Add the missing self-reflection step noted in the code comment above (Day
   4): after `decide_escalation` runs, add a second check that reviews the
   decision against a stated policy (e.g., "never leave a billing issue for
   a paying customer unescalated") and can override the first decision.
2. Add a second tool of your own (Day 2-style: name, description, schema,
   and a real Python function).
3. Replace at least one `# TODO: replace with a real LLM call` with an
   actual Anthropic API call, following the pattern shown in Part 3 above.
4. Test your finished agent on at least 3 different example tickets and
   confirm the routing/escalation decisions make sense.


## 5. DELIVERABLE: Agent workflow documentation

Once your agent is working, write up a short documentation section (as
markdown cells right here in the notebook, or as a separate `.md` file)
covering:

1. **Purpose** — what task does this agent perform, in one sentence?
2. **Tools** — list every tool the agent can call, with its name and a
   one-line description of what it does.
3. **Loop structure** — describe the perceive-reason-act steps in order,
   the way the `print()` statements in `triage_agent` trace them out.
4. **Workflow pattern(s) used** — which of Day 3's five patterns
   (chaining, routing, parallelization, orchestrator-worker,
   evaluator-optimizer) does your agent use, and where?
5. **Reasoning / self-reflection** — where does your agent reason about
   what to do next, and does it include any self-correction step? If not,
   identify one place where adding one would make the agent more robust.
6. **Known limitations** — what inputs or edge cases would break your
   agent, or cause it to make a bad decision? (Every real agent has some —
   documenting them honestly is part of the deliverable, not a sign of
   failure.)

This documentation habit is directly transferable to real-world agent
development, where undocumented agent behavior is a common source of
production incidents.
